# 🧬 06 · 基因组变异与 VCF

> 变异（variation）是基因组学的核心：个体差异、疾病关联、群体遗传都围绕它。
> 本节：SNP/Indel 概念、**VCF 格式解析**、等位基因频率、变异注释与过滤。

## 📋 本节要解决的问题

- SNP 和 Indel 是什么？为什么关心它们？
- VCF 的 8 个固定列 + INFO/GT 怎么读？
- 等位基因频率（AF/MAF）怎么算？
- 变异质量过滤（DP/GQ/FILTER）怎么理解？

## 1️⃣ 变异类型

| 类型 | 全称 | 说明 |
|---|---|---|
| SNP | 单核苷酸多态 | 单个碱基替换（如 A→G） |
| Indel | 插入/缺失 | 小片段插入或缺失，会引起移码 |
| SV | 结构变异 | 大片段缺失/重复/倒位/易位（CNV 是拷贝数变异） |
| CNV | 拷贝数变异 | 基因拷贝数增多或减少 |

> **面试**：SNP 与突变的区别？→ 突变一般指与参考基因组不同（个体内在）；
> SNP 强调在**群体中频率 ≥1%** 的稳定多态位点。

In [1]:
# 模拟参考基因组 + 样本测序比对后的变异
ref = 'ATCGATCGGATCCTAGCTAG'
var_sites = [
    ('chr1', 4, 'G', 'A'),    # SNP
    ('chr1', 7, 'A', 'AAT'),  # 插入 AAT
    ('chr1', 15, 'TC', 'T'),  # 缺失 C
]
print('模拟变异列表:')
for chrom, pos, refb, alt in var_sites:
    print('  %s:%d  ref=%s alt=%s' % (chrom, pos, refb, alt))
print()
print('→ 变异描述三要素：位置 + 参考碱基 + 替代碱基')
assert len(var_sites) == 3
print('✅ 变异概念示例完成')

模拟变异列表:
  chr1:4  ref=G alt=A
  chr1:7  ref=A alt=AAT
  chr1:15  ref=TC alt=T

→ 变异描述三要素：位置 + 参考碱基 + 替代碱基
✅ 变异概念示例完成


## 2️⃣ VCF 格式解析

**VCF**（Variant Call Format）每行 8 列 + 可选样本列：
`CHROM POS ID REF ALT QUAL FILTER INFO`

| 列 | 含义 |
|---|---|
| POS | 1-based 位点 |
| REF | 参考等位基因 |
| ALT | 替代等位基因 |
| QUAL | 变异质量 `Phred` 得分（越高越可信） |
| FILTER | PASS 或失败原因（如 DP<10） |
| INFO | 键值对，如 DP=深度、AF=等位基因频率 |
| 样本列 | GT=基因型（0 参考 / 1 变异，0/1 杂合）

In [2]:
def parse_vcf(text):
    "解析 VCF -> [{chrom,pos,ref,alt,qual,filter,info}]"
    recs = []
    for line in text.splitlines():
        line = line.strip()
        if not line or line.startswith('#'):
            continue
        f = line.split('\t')
        info = {}
        for kv in f[7].split(';'):
            if '=' in kv:
                k, v = kv.split('=', 1)
                info[k] = v
            else:
                info[kv] = True
        recs.append({'chrom': f[0], 'pos': int(f[1]), 'ref': f[3], 'alt': f[4],
                     'qual': float(f[5]), 'filter': f[6], 'info': info})
    return recs

vcf = '''##fileformat=VCFv4.2
#CHROM	POS	ID	REF	ALT	QUAL	FILTER	INFO
chr1	1000	.	G	A	120	PASS	DP=30;AF=0.25
chr1	1005	.	C	T	15	q10	DP=8;AF=0.10
chr2	2000	rs123	G	T	200	PASS	DP=45;AF=0.50'''
recs = parse_vcf(vcf)
print('记录数:', len(recs))
for r in recs:
    print('  %s:%d %s->%s QUAL=%g FILTER=%s AF=%s' %
          (r['chrom'], r['pos'], r['ref'], r['alt'], r['qual'], r['filter'], r['info'].get('AF')))
interesting = [r for r in recs if r['qual'] > 30 and r['filter'] == 'PASS']
print()
print('PASS 且 QUAL>30 的高可信变异数:', len(interesting))
assert len(recs) == 3 and len(interesting) == 2
print('✅ VCF 解析与质量过滤完成')

记录数: 3
  chr1:1000 G->A QUAL=120 FILTER=PASS AF=0.25
  chr1:1005 C->T QUAL=15 FILTER=q10 AF=0.10
  chr2:2000 G->T QUAL=200 FILTER=PASS AF=0.50

PASS 且 QUAL>30 的高可信变异数: 2
✅ VCF 解析与质量过滤完成


## 3️⃣ 手算示例：等位基因频率

> 一个位点，20 个个体（二倍体 2N=40 个等位基因），
> 计数：**AA=12, Aa=6, aa=2**（A 为参考，a 为变异）

**等位基因 a 的频率（频率）**：
$$AF = \frac{\text{a 的拷贝数}}{2N} = \frac{2\times2 + 6}{40} = \frac{10}{40} = 0.25$$

**基因型频率**：AA=12/20=0.60，Aa=6/20=0.30，aa=2/20=0.10
**MAF（较少等位基因频率）** = min(AF, 1-AF) = 0.25（A、a 都不少于 25%）

In [3]:
# 验证手算
aa_hom, het, alt_hom = 12, 6, 2      # AA, Aa, aa
n_ind = aa_hom + het + alt_hom
af = (2*alt_hom + het) / (2*n_ind)
print('等位基因a频率(手算 0.25) =', af)
print('基因型频率: AA=%.2f Aa=%.2f aa=%.2f' % (aa_hom/n_ind, het/n_ind, alt_hom/n_ind))
maf = min(af, 1 - af)
print('MAF(手算 0.25) =', maf)
assert abs(af - 0.25) < 1e-9 and abs(maf - 0.25) < 1e-9
print('✅ 等位基因频率手算验证通过')

等位基因a频率(手算 0.25) = 0.25
基因型频率: AA=0.60 Aa=0.30 aa=0.10
MAF(手算 0.25) = 0.25
✅ 等位基因频率手算验证通过


## 4️⃣ GT 基因型与 Hardy-Weinberg 粗检

样本列 GT：`0/0` 纯合参考、`0/1` 杂合、`1/1` 纯合变异。
> 群体遗传学：Hardy-Weinberg 平衡下 $p^2 + 2pq + q^2 = 1$，
> 可用于质控（严重偏离 → 存在测序错误或选择压力）。

In [4]:
from collections import Counter
# 20 个样本的 GT 列表
gts = ['0/0']*12 + ['0/1']*6 + ['1/1']*2
cnt = Counter(gts)
p = cnt['1/1']*2 + cnt['0/1']          # a 等位基因数
q = cnt['0/0']*2 + cnt['0/1']          # A 等位基因数
fa = p / (p + q)
print('GT 分布:', dict(cnt))
print('a 等位基因频率(手算 0.25) =', fa)
# HW 预期基因型频率
exp_hom_alt = fa**2
exp_het = 2 * fa * (1 - fa)
print('HW 预期: aa=%.2f Aa=%.2f' % (exp_hom_alt, exp_het))
print('观察值 : aa=%.2f Aa=%.2f' % (2/20, 6/20))
assert abs(fa - 0.25) < 1e-9
print('✅ GT 与 HW 粗检完成')

GT 分布: {'0/0': 12, '0/1': 6, '1/1': 2}
a 等位基因频率(手算 0.25) = 0.25
HW 预期: aa=0.06 Aa=0.38
观察值 : aa=0.10 Aa=0.30
✅ GT 与 HW 粗检完成


## ✅ 自测清单

- [ ] 能区分 SNP / Indel / SV / CNV
- [ ] 能解析 VCF 的固定列与 INFO/GT
- [ ] 能手算等位基因频率与 MAF
- [ ] 能理解 QUAL/FILTER/DP 的质量控制意义